# Part 3: NLP and Sequence Modeling Mini Project
## Customer Support Text Classification — Sentiment Analysis

**Dataset:** `customer_support_text_classification.csv`  
**Task:** Classify customer support messages as `positive`, `neutral`, or `negative`  
**Goal:** Build a complete NLP pipeline from raw text to sequence-based deep learning

---

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, precision_score,
    recall_score, f1_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, LSTM, Dense, Dropout,
    Bidirectional, GlobalMaxPooling1D
)
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

print('All imports successful.')
print(f'TensorFlow version: {tf.__version__}')

---
## Task 1: Dataset Understanding

The dataset contains **1,500 customer support messages** sourced from multiple channels (chat, email, phone, social media, app). Each message is labelled with one of three sentiment classes: `positive`, `neutral`, or `negative`.

In [ ]:
df = pd.read_csv('customer_support_text_classification.csv')

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Total Records  : {len(df)}')
print(f'Columns        : {list(df.columns)}')
print(f'\nTarget Column  : sentiment_label')
print(f'Classes        : {df["sentiment_label"].unique().tolist()}')
print(f'\nClass Distribution:')
print(df['sentiment_label'].value_counts())
print(f'\nChannel Breakdown:')
print(df['channel'].value_counts())
print(f'\nUrgent Flag:')
print(df['urgent_flag'].value_counts())
print(f'\nWord Count Statistics:')
print(df['word_count'].describe().round(2))
print(f'\nSample Records:')
for i in range(4):
    print(f"  [{df['sentiment_label'].iloc[i].upper()}] {df['customer_message'].iloc[i]}")
print(f'\nNull values: {df.isnull().sum().sum()}')

In [ ]:
# EDA Visualizations
df['text_length'] = df['customer_message'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Customer Support Dataset — EDA', fontsize=15, fontweight='bold')

palette = {'positive': '#2ecc71', 'neutral': '#3498db', 'negative': '#e74c3c'}

# Class distribution
counts = df['sentiment_label'].value_counts()
bar_colors = [palette[k] for k in counts.index]
axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor='black', linewidth=1.2)
axes[0].set_title('Class Distribution (3-class)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, (val, cnt) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, cnt + 5, str(cnt), ha='center', fontweight='bold')

# Word count by sentiment
for label, color in palette.items():
    subset = df[df['sentiment_label'] == label]['text_length']
    axes[1].hist(subset, bins=20, alpha=0.55, label=label, color=color, edgecolor='black', lw=0.5)
axes[1].set_title('Word Count by Sentiment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Messages by channel
ch_counts = df['channel'].value_counts()
axes[2].bar(ch_counts.index, ch_counts.values, color='#9b59b6', edgecolor='black', linewidth=1.2)
axes[2].set_title('Messages by Channel', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Channel')
axes[2].set_ylabel('Count')
for i, cnt in enumerate(ch_counts.values):
    axes[2].text(i, cnt + 3, str(cnt), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA figure saved.')

**Key Observations:**
- Dataset has **1,500 records** across 3 balanced sentiment classes: neutral (524), negative (497), positive (479)
- Messages come from 5 channels: email, social, phone, chat, app
- Average message length is ~12.7 words — short and concise customer support queries
- 400 out of 1,500 messages are flagged as urgent
- No missing values — clean dataset ready for NLP processing

---
## Task 2: Text Preprocessing

Raw text must be cleaned before vectorization. The preprocessing pipeline applied:

1. **Lowercasing** — normalize all text to lowercase
2. **HTML tag removal** — remove any `<br />`, `<b>` etc. that may appear in scraped text
3. **URL removal** — strip any hyperlinks
4. **Special character & digit removal** — keep only alphabetic tokens
5. **Tokenization** — split text on whitespace
6. **Stopword removal** — remove common English function words that carry no sentiment signal

In [ ]:
# Hardcoded stopwords (no NLTK download needed)
STOPWORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y','ain','aren','couldn','didn','doesn','hadn',
    'hasn','haven','isn','ma','mightn','mustn','needn','shan','shouldn',
    'wasn','weren','won','wouldn','would','could','might','shall','may',
    'please','ticket','number','need','information'
}

def preprocess_text(text):
    """Full NLP preprocessing pipeline for customer support messages."""
    text = str(text).lower()                       # 1. Lowercase
    text = re.sub(r'<[^>]+>', '', text)            # 2. Remove HTML tags
    text = re.sub(r'http\S+|www\S+', '', text)     # 3. Remove URLs
    text = re.sub(r'[^a-z\s]', '', text)           # 4. Remove special chars / digits
    text = re.sub(r'\s+', ' ', text).strip()       # 5. Normalize whitespace
    tokens = text.split()                          # 6. Tokenize
    tokens = [t for t in tokens                    # 7. Remove stopwords + single chars
              if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
df['cleaned_length']  = df['cleaned_message'].apply(lambda x: len(x.split()))

print('Preprocessing complete!')
print(f'Avg original length : {df["text_length"].mean():.1f} words')
print(f'Avg cleaned length  : {df["cleaned_length"].mean():.1f} words')
print(f'\nExample — Original : {df["customer_message"].iloc[0]}')
print(f'Example — Cleaned  : {df["cleaned_message"].iloc[0]}')
print(f'\nExample — Original : {df["customer_message"].iloc[2]}')
print(f'Example — Cleaned  : {df["cleaned_message"].iloc[2]}')

---
## Task 3: Text Vectorization

### Why must text be converted to vectors?

Machine learning models are mathematical functions — they **cannot directly process strings**. Text must be mapped to a numerical space so that:
- Mathematical operations (matrix multiplication, gradient descent) can be applied
- Relationships between words can be measured via distance/similarity
- Patterns distinguishing classes can be learned

### Three vectorization approaches applied:

| Method | Representation | Pros | Cons |
|---|---|---|---|
| **Bag of Words** | Word count per document | Simple, interpretable | Ignores order; sparse |
| **TF-IDF** | Word importance weighted by rarity | Reduces common-word noise | Still ignores word order |
| **Embedding + Tokenizer** | Dense learned vector per word | Captures semantics; order preserved | Needs more data & compute |

In [ ]:
X = df['cleaned_message']
y = df['sentiment_label']

# Label encoding: negative=0, neutral=1, positive=2
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
print(f'\nTrain: {len(X_train)} samples | Test: {len(X_test)} samples')

# ── Bag of Words ──
bow_vec = CountVectorizer(max_features=500)
X_train_bow = bow_vec.fit_transform(X_train)
X_test_bow  = bow_vec.transform(X_test)
print(f'\nBoW matrix shape (train)   : {X_train_bow.shape}')
print(f'  → Each document = {X_train_bow.shape[1]}-d sparse count vector')

# ── TF-IDF (unigrams + bigrams) ──
tfidf_vec = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)
print(f'\nTF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'  → Includes unigrams and bigrams; weights by inverse document frequency')

# ── Keras Tokenizer + Padded Sequences (for LSTM) ──
MAX_VOCAB = 500
MAX_LEN   = 20   # messages avg ~12 words; 20 covers 95%+
EMBED_DIM = 32

tokenizer = KerasTokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'\nPadded sequence shape (train): {X_train_pad.shape}')
print(f'  → Each message = fixed-length integer ID sequence of length {MAX_LEN}')
print(f'\nSample message     : "{X_train.values[0]}"')
print(f'After tokenization : {X_train_pad[0]}')

---
## Task 4: Baseline Models

Two classical machine learning baselines are trained before the deep learning approach:

1. **Logistic Regression + TF-IDF** — linear classifier on weighted word-frequency features
2. **Multinomial Naive Bayes + Bag of Words** — probabilistic model based on word counts

Both are multiclass classifiers (negative / neutral / positive). Evaluation uses accuracy, precision, recall, F1-score, and confusion matrices.

In [ ]:
# ── Model 1: Logistic Regression + TF-IDF ──
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)

print('Logistic Regression + TF-IDF')
print('=' * 50)
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

# ── Model 2: Naive Bayes + BoW ──
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)
nb_acc = accuracy_score(y_test, y_pred_nb)

print('\nNaive Bayes + Bag of Words')
print('=' * 50)
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

print(f'LR Accuracy : {lr_acc:.4f}')
print(f'NB Accuracy : {nb_acc:.4f}')

In [ ]:
# Evaluation plots — Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline Model Evaluation — Confusion Matrices', fontsize=14, fontweight='bold')

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[0].set_title(f'Logistic Regression + TF-IDF\nAccuracy: {lr_acc:.4f}', fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

cm_nb = confusion_matrix(y_test, y_pred_nb)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[1].set_title(f'Naive Bayes + BoW\nAccuracy: {nb_acc:.4f}', fontweight='bold')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('results/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# Save metrics CSV
results_df = pd.DataFrame({
    'Model'     : ['Logistic Regression + TF-IDF', 'Naive Bayes + BoW'],
    'Accuracy'  : [round(lr_acc, 4), round(nb_acc, 4)],
    'Precision' : [round(precision_score(y_test, y_pred_lr, average='weighted'), 4),
                   round(precision_score(y_test, y_pred_nb, average='weighted'), 4)],
    'Recall'    : [round(recall_score(y_test, y_pred_lr, average='weighted'), 4),
                   round(recall_score(y_test, y_pred_nb, average='weighted'), 4)],
    'F1'        : [round(f1_score(y_test, y_pred_lr, average='weighted'), 4),
                   round(f1_score(y_test, y_pred_nb, average='weighted'), 4)],
})
results_df.to_csv('results/model_evaluation.csv', index=False)
print(results_df.to_string(index=False))

---
## Task 5: Sequence Model — Bidirectional LSTM

### Architecture Design

| Layer | Configuration | Purpose |
|---|---|---|
| **Input** | Integer sequences, max_len = 20 | One token ID per time step |
| **Embedding** | vocab = 500, dim = 32, trainable | Converts word IDs to dense semantic vectors |
| **BiLSTM** | 32 units forward + 32 backward, return_sequences=True | Captures context in both directions |
| **GlobalMaxPooling1D** | – | Extracts the most salient feature across all time steps |
| **Dense** | 32 units, ReLU activation | Non-linear feature combination |
| **Dropout** | rate = 0.4 | Regularization to prevent overfitting |
| **Output** | 3 units, Softmax activation | Probability over 3 sentiment classes |

**Loss function:** Categorical Cross-Entropy (standard for multiclass classification)  
**Optimizer:** Adam  
**Metric:** Accuracy

In [ ]:
NUM_CLASSES = 3
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

tf.random.set_seed(42)

model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(32, return_sequences=True, dropout=0.3)),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_test_pad, y_test_cat),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate BiLSTM
y_pred_prob = model.predict(X_test_pad, verbose=0)
y_pred_lstm = np.argmax(y_pred_prob, axis=1)
lstm_acc = accuracy_score(y_test, y_pred_lstm)

print(f'BiLSTM Test Accuracy: {lstm_acc:.4f}')
print(classification_report(y_test, y_pred_lstm, target_names=le.classes_))

In [ ]:
# Training history plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Bidirectional LSTM — Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['accuracy'],     'b-o', label='Train', lw=2, markersize=4)
axes[0].plot(history.history['val_accuracy'], 'r-o', label='Val',   lw=2, markersize=4)
axes[0].set_title('Model Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     'b-o', label='Train', lw=2, markersize=4)
axes[1].plot(history.history['val_loss'], 'r-o', label='Val',   lw=2, markersize=4)
axes[1].set_title('Model Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample predictions
samples     = X_test.values[:10]
true_labels = [le.classes_[t] for t in y_test[:10]]

pred_seqs   = tokenizer.texts_to_sequences(samples)
pred_pad    = pad_sequences(pred_seqs, maxlen=MAX_LEN, padding='post', truncating='post')
pred_probs  = model.predict(pred_pad, verbose=0)
pred_ids    = np.argmax(pred_probs, axis=1)
pred_labels = [le.classes_[p] for p in pred_ids]

print('SAMPLE PREDICTIONS')
print('=' * 70)
with open('results/sample_predictions.txt', 'w') as f:
    f.write('SAMPLE PREDICTIONS — BiLSTM Model\n')
    f.write('Dataset: Customer Support Text Classification\n')
    f.write('=' * 80 + '\n\n')
    for i, (text, true, pred, probs) in enumerate(zip(samples, true_labels, pred_labels, pred_probs)):
        conf = max(probs)
        status = '✓ CORRECT' if true == pred else '✗ WRONG'
        line = f'Sample {i+1}: {status} | True: {true} | Pred: {pred} (conf: {conf:.4f})'
        print(line)
        f.write(f'Sample {i+1}: {status}\n')
        f.write(f'Text      : {text}\n')
        f.write(f'True      : {true}\n')
        f.write(f'Predicted : {pred}  (confidence: {conf:.4f})\n')
        f.write(f'All probs : neg={probs[0]:.3f} | neu={probs[1]:.3f} | pos={probs[2]:.3f}\n\n')
print('\nSample predictions saved to results/sample_predictions.txt')

In [ ]:
# Final model comparison
fig, ax = plt.subplots(figsize=(9, 5))
models_names = ['Logistic Reg\n(TF-IDF)', 'Naive Bayes\n(BoW)', 'BiLSTM\n(Embeddings)']
accs = [lr_acc, nb_acc, lstm_acc]
bar_colors = ['#3498db', '#e67e22', '#e74c3c']

bars = ax.bar(models_names, [a * 100 for a in accs],
              color=bar_colors, edgecolor='black', linewidth=1.2, width=0.5)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison\nCustomer Support Sentiment Classification (3-class)',
             fontsize=12, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{acc * 100:.2f}%', ha='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Task 6: Attention and Transformer Reflection

### Why RNNs Struggle with Long-Term Dependencies

Recurrent Neural Networks (RNNs) process text **one token at a time**, maintaining a hidden state `h_t` that is passed forward:

```
h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)
```

The problem: gradients are **backpropagated through time** by repeatedly multiplying the same weight matrix `W_h`. If the largest eigenvalue of `W_h` is less than 1, gradients **shrink exponentially** toward zero as they propagate backwards (vanishing gradients). Early tokens in the sequence have negligible influence on the final prediction. A customer message like *"The service started well but the resolution was completely unacceptable"* may lose the early positive signal by the time the RNN processes "unacceptable".

---

### How LSTMs Help with Memory

LSTMs (Hochreiter & Schmidhuber, 1997) introduce a **cell state** `C_t` — a "conveyor belt" that passes information across time steps with only minor, gated modifications:

```
Forget gate : f_t = σ(W_f · [h_{t-1}, x_t] + b_f)   ← what to erase
Input gate  : i_t = σ(W_i · [h_{t-1}, x_t] + b_i)   ← what to store
Cell update : C_t = f_t ⊙ C_{t-1} + i_t ⊙ tanh(W_c · [h_{t-1}, x_t])
Output gate : o_t = σ(W_o · [h_{t-1}, x_t] + b_o)   ← what to expose
```

Because the cell state is modified by **addition** (not multiplication), gradients can flow backwards without exponential decay. LSTMs can remember relevant context across hundreds of time steps — crucial for longer customer messages or multi-sentence tickets.

---

### What Attention Solves in Sequence-to-Sequence Tasks

Even with LSTMs, sequence-to-sequence tasks (translation, summarization) hit a **bottleneck**: the encoder must compress an entire variable-length input into a single fixed-size vector before decoding begins. For long inputs, this compresses too much and loses detail.

**Attention** (Bahdanau et al., 2015) lets the decoder **dynamically query all encoder hidden states** at each decoding step:

```
score(s_t, h_i) = v^T · tanh(W_1·s_t + W_2·h_i)     # alignment score
α_i = softmax(score(s_t, h_i))                        # attention weights
context = Σ α_i · h_i                                 # weighted encoder states
```

This solves the bottleneck — the decoder can attend to whichever part of the input is most relevant at each generation step, enabling much better alignment between input and output.

---

### Why Transformers Are Important in Modern NLP and Generative AI

The **Transformer** (Vaswani et al., 2017 — "Attention Is All You Need") removed recurrence entirely, replacing it with **self-attention** — every token simultaneously attends to every other token:

```
Attention(Q, K, V) = softmax(QK^T / √d_k) · V
```

Key advantages over RNNs and LSTMs:

| Property | RNN / LSTM | Transformer |
|---|---|---|
| Parallelism | Sequential — must wait for `h_{t-1}` | Fully parallel across all positions |
| Long-range dependency | Degrades with sequence length | Constant-distance attention to any position |
| Training speed | Slow (sequential ops) | Fast (GPU/TPU parallelism) |
| Scalability | Saturates | Scales with model size + data |

Transformers enabled pre-training on internet-scale data. This led directly to **BERT** (bi-directional encoder for classification/NER), **GPT** (autoregressive decoder for generation), **T5**, and today's large language models — Claude, ChatGPT, Gemini. Every modern Generative AI system is built on transformer architectures, making understanding transformers essential for anyone working in NLP.

---

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Dataset             : Customer Support Text Classification')
print(f'Records             : 1,500 | Classes: 3 (positive / neutral / negative)')
print()
print(f'Logistic Regression (TF-IDF) : Accuracy = {lr_acc:.4f}')
print(f'Naive Bayes (BoW)            : Accuracy = {nb_acc:.4f}')
print(f'Bidirectional LSTM           : Accuracy = {lstm_acc:.4f}')
print()
print('Outputs saved to results/')
print('  eda_analysis.png          Task 1')
print('  model_evaluation.png      Task 4 — confusion matrices')
print('  model_evaluation.csv      Task 4 — metrics table')
print('  lstm_training_history.png Task 5 — training curves')
print('  lstm_architecture.png     Task 5 — architecture diagram')
print('  model_comparison.png      All models compared')
print('  sample_predictions.txt    Task 5 — 10 sample predictions')